# Exploratory Data Analysis of Housing Dataset




## 1. Data loading


In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

if not os.path.exists("train.csv"):
    try:
        from google.colab import files
        print("Please upload train.csv")
        files.upload()
    except ImportError:
        raise FileNotFoundError("train.csv not found in the working directory.")

df = pd.read_csv("train.csv")

print("Shape (rows, columns):", df.shape)

In [ ]:
print("First 5 rows:")
display(df.head())

print("\nInfo (column names, dtypes, non-null counts):")
df.info()

missing = df.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]
missing_report = pd.DataFrame({
    "missing_count": missing,
    "missing_pct": (missing / len(df) * 100).round(2)
})
print(f"\nColumns with missing values: {len(missing_report)} out of {df.shape[1]}")
display(missing_report)

## 2. Data understanding


In [ ]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()
categorical_cols = df.select_dtypes(include="object").columns.tolist()

ordinal_map = {
    "ExterQual":   ["Po", "Fa", "TA", "Gd", "Ex"],
    "KitchenQual": ["Po", "Fa", "TA", "Gd", "Ex"],
}
for col, order in ordinal_map.items():
    df[col] = pd.Categorical(df[col], categories=order, ordered=True)

ordinal_numeric = ["OverallQual", "OverallCond"]
nominal_numeric = ["MSSubClass"]

def classify(col):
    if col in ordinal_map or col in ordinal_numeric:
        return "Ordinal"
    if col in categorical_cols or col in nominal_numeric:
        return "Categorical (nominal)"
    s = df[col].dropna()
    is_int = np.allclose(s, s.round())
    if is_int and (("Yr" in col) or ("Year" in col) or s.nunique() <= 20):
        return "Numerical (discrete)"
    return "Numerical (continuous)"

summary_table = pd.DataFrame({
    "column": df.columns,
    "type": [classify(c) for c in df.columns],
    "n_unique": [df[c].nunique() for c in df.columns],
    "example_values": [df[c].dropna().unique()[:3].tolist() for c in df.columns],
})
display(summary_table)
print("\nCount of columns by type:")
print(summary_table["type"].value_counts())

In [ ]:
print("Describe (numeric columns):")
display(df.describe().T)

rows = []
for col in ["SalePrice", "GrLivArea", "LotArea"]:
    s = df[col]
    mean, median = s.mean(), s.median()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    if mean > median:
        note = "right skewed (mean > median)"
    elif mean < median:
        note = "left skewed (mean < median)"
    else:
        note = "roughly symmetric"
    rows.append({
        "column": col, "mean": mean, "median": median, "mode": s.mode()[0],
        "std": s.std(), "variance": s.var(), "range": s.max() - s.min(),
        "iqr": q3 - q1, "skewness": s.skew(), "kurtosis": s.kurt(), "note": note,
    })
stats_table = pd.DataFrame(rows).set_index("column")
pd.options.display.float_format = "{:,.2f}".format
display(stats_table.T)
pd.reset_option("display.float_format")

gap = df["SalePrice"].mean() - df["SalePrice"].median()
print(f"\nSalePrice: mean - median = {gap:,.0f}")

for col in ["Neighborhood", "HouseStyle"]:
    print(f"\nValue counts for {col} (mode = {df[col].mode()[0]}):")
    print(df[col].value_counts())

## 3. Data cleaning


In [ ]:
# duplicates
dup_count = df.duplicated().sum()
dup_count_no_id = df.drop(columns="Id").duplicated().sum()
print("Duplicate rows:", dup_count, "| Duplicates ignoring Id:", dup_count_no_id)
if dup_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print("New shape:", df.shape)

# inconsistent categories
# Check casing / extra spaces
problem_cols = []
for col in categorical_cols:
    vals = df[col].dropna().astype(str).unique()
    stripped = [v.strip() for v in vals]
    if any(v != s for v, s in zip(vals, stripped)) or \
       len({s.lower() for s in stripped}) != len(set(stripped)):
        problem_cols.append(col)
print("Columns with casing/space problems:", problem_cols if problem_cols else "none")
for col in problem_cols:
    df[col] = df[col].astype(str).str.strip()

# Similar spellings: some labels differ slightly from the official data description
spelling_fix = {
    "Exterior1st": {"BrkComm": "Brk Cmn", "CmentBd": "CemntBd", "WdShing": "Wd Shng"},
    "Exterior2nd": {"BrkComm": "Brk Cmn", "CmentBd": "CemntBd", "WdShing": "Wd Shng"},
    "BldgType":    {"2fmCon": "2FmCon", "Duplex": "Duplx", "Twnhs": "TwnhsI"},
}
for col, mapping in spelling_fix.items():
    before = df[col].nunique()
    df[col] = df[col].replace(mapping)
    print(f"{col}: unique labels {before} -> {df[col].nunique()}")

In [ ]:
# Missing values
# (i) NA means "feature absent" -> replace with "None" (not unknown, not dropped)
none_cols = ["PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
             "GarageType", "GarageFinish", "GarageQual", "GarageCond",
             "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
             "MasVnrType"]
for col in none_cols:
    if col in df.columns and col not in ordinal_map:
        df[col] = df[col].fillna("None")

# (ii) Numeric columns
# LotFrontage: median of the same Neighborhood (similar lots), then overall median as fallback
df["LotFrontage"] = df.groupby("Neighborhood")["LotFrontage"].transform(lambda s: s.fillna(s.median()))
df["LotFrontage"] = df["LotFrontage"].fillna(df["LotFrontage"].median())
# MasVnrArea: no veneer -> 0
df["MasVnrArea"] = df["MasVnrArea"].fillna(0)
# GarageYrBlt: no garage -> use YearBuilt
df["GarageYrBlt"] = df["GarageYrBlt"].fillna(df["YearBuilt"])

# (iii) Anything else still missing: numeric -> median, text -> most common value
for col in df.columns[df.isnull().any()]:
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].fillna(df[col].median())
    else:
        df[col] = df[col].fillna(df[col].mode()[0])
        print(f"Filled remaining nulls in {col} with its mode")

# Verify
print("\nRemaining missing values:", int(df.isnull().sum().sum()))

# Impossible values
area_cols = [c for c in df.columns if ("SF" in c) or ("Area" in c)]
print("\nYearBuilt > YrSold:", int((df["YearBuilt"] > df["YrSold"]).sum()))
print("YearRemodAdd < YearBuilt:", int((df["YearRemodAdd"] < df["YearBuilt"]).sum()))
print("GarageYrBlt > YrSold:", int((df["GarageYrBlt"] > df["YrSold"]).sum()))
print("Negative area values:", int((df[area_cols] < 0).any(axis=1).sum()))

## 4. Univariate analysis (Step 4: Distributions)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df["SalePrice"], kde=True, ax=axes[0])
axes[0].set_title("SalePrice distribution")
axes[0].set_xlabel("SalePrice ($)"); axes[0].set_ylabel("Count")

sns.boxplot(y=df["SalePrice"], ax=axes[1])
axes[1].set_title("SalePrice boxplot")
axes[1].set_ylabel("SalePrice ($)")

sns.histplot(np.log(df["SalePrice"]), kde=True, ax=axes[2])
axes[2].set_title("log(SalePrice) distribution")
axes[2].set_xlabel("log(SalePrice)"); axes[2].set_ylabel("Count")

plt.tight_layout()
plt.show()

print(f"Skewness before log: {stats.skew(df['SalePrice']):.3f}")
print(f"Skewness after log : {stats.skew(np.log(df['SalePrice'])):.3f}")

In [ ]:
# Boxplot of SalePrice by OverallQual
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x="OverallQual", y="SalePrice")
plt.title("SalePrice by OverallQual")
plt.xlabel("Overall quality (1 = very poor, 10 = excellent)"); plt.ylabel("SalePrice ($)")
plt.show()

print("Median price by OverallQual:")
print(df.groupby("OverallQual")["SalePrice"].median())

# Boxplot of SalePrice by Neighborhood, sorted by median
order = df.groupby("Neighborhood")["SalePrice"].median().sort_values().index
plt.figure(figsize=(14, 6))
sns.boxplot(data=df, x="Neighborhood", y="SalePrice", order=order)
plt.xticks(rotation=90)
plt.title("SalePrice by Neighborhood (sorted by median)")
plt.xlabel("Neighborhood"); plt.ylabel("SalePrice ($)")
plt.show()

print("Top 5 neighborhoods by median price:")
print(df.groupby("Neighborhood")["SalePrice"].median().sort_values(ascending=False).head(5))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.countplot(data=df, x="OverallQual", ax=axes[0])
axes[0].set_title("Count of houses by OverallQual")
axes[0].set_xlabel("OverallQual"); axes[0].set_ylabel("Count")

sns.countplot(data=df, x="HouseStyle", order=df["HouseStyle"].value_counts().index, ax=axes[1])
axes[1].set_title("Count of houses by HouseStyle")
axes[1].set_xlabel("HouseStyle"); axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## 5. Bivariate and multivariate analysis


In [ ]:
num_df = df.select_dtypes(include="number").drop(columns=["Id"])

cov_matrix = num_df.cov()
corr_matrix = num_df.corr(method="pearson")

top_corr = corr_matrix["SalePrice"].drop("SalePrice").abs().sort_values(ascending=False)
top10 = top_corr.head(10).index.tolist()

print("Top 10 features by |correlation| with SalePrice:")
print(corr_matrix.loc[top10, "SalePrice"].round(3))

print("\nCovariance vs correlation with SalePrice for the top 3 features:")
for feat in top10[:3]:
    print(f"{feat:12s} covariance = {cov_matrix.loc[feat, 'SalePrice']:>18,.1f}   "
          f"correlation = {corr_matrix.loc[feat, 'SalePrice']:.3f}")

print("\nWhy correlation is easier to compare: covariance depends on the units/scale of the two variables "
      "(sq. feet x dollars vs quality points x dollars), so its size cannot be compared across features. "
      "Correlation is standardised to the range -1 to +1.")

# Optional: Spearman correlation for ordinal columns
ordinal_df = pd.DataFrame({
    "OverallQual": df["OverallQual"],
    "ExterQual": df["ExterQual"].cat.codes,
    "KitchenQual": df["KitchenQual"].cat.codes,
    "SalePrice": df["SalePrice"],
})
print("\nSpearman correlation of ordinal columns with SalePrice:")
print(ordinal_df.corr(method="spearman")["SalePrice"].drop("SalePrice").round(3))

# Feature pairs that are strongly correlated with each other
pairs = corr_matrix.where(np.triu(np.ones(corr_matrix.shape, dtype=bool), k=1)).stack()
strong_pairs = pairs[pairs.abs() > 0.8].sort_values(ascending=False)
print("\nFeature pairs with |correlation| > 0.8:")
print(strong_pairs.round(3))
print("\nGarageArea vs GarageCars correlation:", round(df["GarageArea"].corr(df["GarageCars"]), 3))

In [ ]:
unusual = (df["GrLivArea"] > 4000) & (df["SalePrice"] < 300000)

# GrLivArea vs SalePrice with unusual points highlighted
plt.figure(figsize=(10, 6))
plt.scatter(df.loc[~unusual, "GrLivArea"], df.loc[~unusual, "SalePrice"], alpha=0.5, label="Normal sales")
plt.scatter(df.loc[unusual, "GrLivArea"], df.loc[unusual, "SalePrice"], color="red", s=80, label="GrLivArea > 4000 and low price")
for _, r in df[unusual].iterrows():
    plt.annotate(f"Id {int(r['Id'])}", (r["GrLivArea"], r["SalePrice"]), textcoords="offset points", xytext=(8, 8))
plt.title("GrLivArea vs SalePrice")
plt.xlabel("Above-ground living area (sq ft)"); plt.ylabel("SalePrice ($)")
plt.legend()
plt.show()

# TotalBsmtSF and YearBuilt vs SalePrice
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.scatterplot(data=df, x="TotalBsmtSF", y="SalePrice", alpha=0.5, ax=axes[0])
axes[0].set_title("TotalBsmtSF vs SalePrice")
axes[0].set_xlabel("Total basement area (sq ft)"); axes[0].set_ylabel("SalePrice ($)")
sns.scatterplot(data=df, x="YearBuilt", y="SalePrice", alpha=0.5, ax=axes[1])
axes[1].set_title("YearBuilt vs SalePrice")
axes[1].set_xlabel("Year built"); axes[1].set_ylabel("SalePrice ($)")
plt.tight_layout()
plt.show()

In [ ]:
selected = ["SalePrice", "GrLivArea", "TotalBsmtSF", "GarageArea", "YearBuilt"]

# Pair plot, diagonal as KDE, coloured by CentralAir (optional)
g = sns.pairplot(df[selected + ["CentralAir"]], hue="CentralAir", diag_kind="kde",
                 plot_kws={"alpha": 0.5, "s": 20}, corner=False)
g.fig.suptitle("Pair plot of key variables", y=1.02)
plt.show()

In [ ]:
corr_top = df[top10 + ["SalePrice"]].corr()
mask = np.triu(np.ones_like(corr_top, dtype=bool))

plt.figure(figsize=(11, 9))
sns.heatmap(corr_top, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            linewidths=0.5, cbar_kws={"label": "Pearson correlation"})
plt.title("Top 10 features correlated with SalePrice")
plt.show()

## 6. Outlier analysis


In [ ]:
def iqr_outliers(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return series[(series < lower) | (series > upper)]

def zscore_outliers(series, threshold=3):
    z = (series - series.mean()) / series.std()
    return series[z.abs() > threshold]

flags = {}
for col in ["GrLivArea", "SalePrice"]:
    a = iqr_outliers(df[col])
    b = zscore_outliers(df[col])
    overlap = a.index.intersection(b.index)
    flags[col] = (a.index, b.index)
    print(f"{col:10s} IQR outliers: {len(a):4d} | z-score outliers: {len(b):4d} | overlap: {len(overlap):4d}")

# Boxplots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col in zip(axes, ["GrLivArea", "SalePrice"]):
    sns.boxplot(y=df[col], ax=ax, flierprops={"markerfacecolor": "red", "markeredgecolor": "red"})
    ax.set_title(f"Boxplot of {col} (outliers in red)")
    ax.set_ylabel(col)
plt.tight_layout()
plt.show()

# Scatter with flagged points
iqr_idx = flags["GrLivArea"][0].union(flags["SalePrice"][0])
z_idx = flags["GrLivArea"][1].union(flags["SalePrice"][1])
plt.figure(figsize=(10, 6))
plt.scatter(df["GrLivArea"], df["SalePrice"], alpha=0.35, label="All points")
plt.scatter(df.loc[iqr_idx, "GrLivArea"], df.loc[iqr_idx, "SalePrice"], facecolors="none",
            edgecolors="orange", s=80, label="Flagged by IQR")
plt.scatter(df.loc[z_idx, "GrLivArea"], df.loc[z_idx, "SalePrice"], color="red", s=20, label="Flagged by z-score")
plt.title("GrLivArea vs SalePrice with flagged outliers")
plt.xlabel("GrLivArea (sq ft)"); plt.ylabel("SalePrice ($)")
plt.legend()
plt.show()

In [ ]:
# very large houses (> 4000 sq ft) that sold very cheaply look like non-typical sales
# (e.g. partial sales), so they are removed. Expensive houses that are also large follow the
# trend and are real sales, so they are KEPT.
remove_mask = (df["GrLivArea"] > 4000) & (df["SalePrice"] < 300000)
print("Rows removed:", int(remove_mask.sum()))
print(df.loc[remove_mask, ["Id", "GrLivArea", "SalePrice", "SaleCondition", "Neighborhood"]])

df_clean = df[~remove_mask].copy()
print("\nShape before:", df.shape, "| after:", df_clean.shape)

print("\nCorrelation GrLivArea vs SalePrice")
print(f"  before removal: {df['GrLivArea'].corr(df['SalePrice']):.3f}")
print(f"  after removal : {df_clean['GrLivArea'].corr(df_clean['SalePrice']):.3f}")

## 7. Dashboard


In [ ]:
main_color = "#2E6F9E"
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel 1: average price by YearBuilt (Matplotlib line plot)
yearly = df_clean.groupby("YearBuilt")["SalePrice"].mean()
axes[0, 0].plot(yearly.index, yearly.values, color=main_color, label="Average price")
axes[0, 0].set_title("Average SalePrice by year built")
axes[0, 0].set_xlabel("Year built"); axes[0, 0].set_ylabel("Average SalePrice ($)")
axes[0, 0].legend()

# Panel 2: top 10 neighborhoods by mean price (Matplotlib horizontal bars)
top_nb = df_clean.groupby("Neighborhood")["SalePrice"].mean().sort_values(ascending=False).head(10).sort_values()
axes[0, 1].barh(top_nb.index, top_nb.values, color=main_color)
axes[0, 1].set_title("Top 10 neighborhoods by mean SalePrice")
axes[0, 1].set_xlabel("Mean SalePrice ($)"); axes[0, 1].set_ylabel("Neighborhood")

# Panel 3: distribution with mean and median (Seaborn)
sns.histplot(df_clean["SalePrice"], kde=True, color=main_color, ax=axes[1, 0])
axes[1, 0].axvline(df_clean["SalePrice"].mean(), color="red", linestyle="--", label="Mean")
axes[1, 0].axvline(df_clean["SalePrice"].median(), color="green", linestyle="-", label="Median")
axes[1, 0].set_title("SalePrice distribution")
axes[1, 0].set_xlabel("SalePrice ($)"); axes[1, 0].set_ylabel("Count")
axes[1, 0].legend()

# Panel 4: GrLivArea vs SalePrice coloured by OverallQual (Seaborn scatter)
sns.scatterplot(data=df_clean, x="GrLivArea", y="SalePrice", hue="OverallQual",
                palette="viridis", alpha=0.7, ax=axes[1, 1])
axes[1, 1].set_title("GrLivArea vs SalePrice (colour = OverallQual)")
axes[1, 1].set_xlabel("GrLivArea (sq ft)"); axes[1, 1].set_ylabel("SalePrice ($)")
axes[1, 1].legend(title="OverallQual", fontsize=8, ncol=2)

fig.suptitle("House Price EDA Summary", fontsize=18, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig("eda_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()

# Download the image from Colab (ignored outside Colab)
try:
    from google.colab import files
    files.download("eda_dashboard.png")
except ImportError:
    pass